# Generate the total age histograms in the stacked figure format

This was figure S6 in the initial submission

In [1]:
from __future__ import print_function

import matplotlib
matplotlib.use('pdf')
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs

#print(os.getcwd())

1664296769.3587751
all_fwctb
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv


In [2]:
plt.show()

In [3]:
figure_output_dir='/Users/BenKaiser/Desktop/'
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/ApJ_reformat/figures'

In [4]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations'
os.chdir(target_dir)

In [5]:

npoints=100
color_dict={
    'GaiaJ1644-0449':'#ff0000',
    'SDSSJ1330+6435':'#8900ff',
    'WDJ2356-209':'#00ffc5'#'#b0ff00'
}

fill_dict={
    'GaiaJ1644-0449':'#ffc3c3',#'#ff8282',
    'SDSSJ1330+6435':'#d0a3ff',#'#c689fb',
    'WDJ2356-209':'#ebfffb'#'#d4fb7e'
}

pattern_dict={
    'GaiaJ1644-0449':'#ffc3c3',
    'SDSSJ1330+6435':'#d0a3ff',
    'WDJ2356-209':'#ebfffb'
}
step_dict={
    'GaiaJ1644-0449':10,
    'SDSSJ1330+6435':0.1,
    'WDJ2356-209':10
}
pos_dict={
    'GaiaJ1644-0449':0,
    'SDSSJ1330+6435':5,
    'WDJ2356-209':10 #'#b0ff00'
}

ssp_pos=2 #offset in addition to object offset for the case of steady-state

#making the points be in the middle of the line
for pos in pos_dict:
    pos_dict[pos]=int(npoints/2.+pos_dict[pos])



line_dict={
    'thin': ':',
    'thick':'-',
    'halo':'-.'
}

wd_marker='*'
met_marker='D'
met_color='#1ca1f2'
ci_size=14
wd_size=14
ci_leg_size=9
dp_alpha=0.5
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
arrow_line=4
letter_off=-3

star_marker='o'
pop_colors=['darkorange','brown','navy','grey'] #thin disk, thick disk, halo, in-between for Bensby plots

spite_alpha=1




In [6]:
#J1644_file='GaiaJ1644m0449_tot_age_MC.csv'
#J1330_file='SDSSJ1330p6435_tot_age_MC.csv'
#J2356_file='1644597624_WDJ2317p1830_tot_age_MC_merged.csv'
#J2356_file='1644603472_WD_J1644m0449_tot_age_MC_merged.csv'
#J2356_file='1644604495_SDSSJ1330p6435_tot_age_MC_merged.csv'
#J2356_file='1644605479_WDJ2356m209_tot_age_MC_merged.csv'
#J2356_file='1644609771_LHS2534_tot_age_MC_merged.csv'
#wd_abund_file='temp_wd_abundances.csv'
#wd_abund_file='20220214_all_wd_abundances_age_distributions.csv'
#wd_abund_file='20220222_all_wd_abundances_newMC_ages.csv'
#wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'
wd_abund_file='20220926_all_wd_abundances_7030MCages.csv'

In [7]:
#J1644_mc= np.genfromtxt(J1644_file)
#J1330_mc= np.genfromtxt(J1330_file)
#J2356_mc= np.genfromtxt(J2356_file)

In [8]:
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table.add_index('name')
wd_inbounds=np.where(wd_abund_table['show_age_dist']==1)
wd_abund_table=wd_abund_table[wd_inbounds]
num_plots=len(wd_abund_table)
print(num_plots)

6


In [9]:
def plot_ml(row, ypos=0.95):
    age_errs= [[row['age_minus']],[row['age_plus']]]
    plt.errorbar(float(row['age']),ypos,xerr=age_errs,label=fs.fix_display_string(row['name']),marker=wd_marker, color=row['plot_color'],markersize=wd_size, linestyle='None')
    return

In [10]:
os.chdir(target_dir)
ypos=0.90
#text_pos= [0.5,0.8]
#text_pos=[3.5,ypos]
#text_pos=[10.25,ypos-0.2]
#text_pos=[10.25,1.05]

##Just old WD's window
#xlims=[6,14]
#full_x_ticklabels=[6,8,10,12,14]
#text_pos=[10.25,1.05]


###including GD378 and young white dwarfs
#xlims=[0,14]
#full_x_ticklabels=[0,2,4,6,8,10,12,14]
#text_pos=[1.25,1.05]


#20Gyr bounds
xlims=[0,20]
full_x_ticklabels=[0,5,10,15,20]
#text_pos=[1.25,1.05] 





#trunc_ticklabels=[0,0.2,0.4,0.6,0.8,1.0]
#full_ticklabels=[0,0.2,0.4,0.6,0.8,1.0,1.2]

trunc_ticklabels=[0,0.5,1.0]
full_ticklabels=[0,0.5,1.0]


#full_x_ticklabels=[6,8,10,12,14]


#trunc_ticklabels=[0,'',0.4,'',0.8,'']
#full_ticklabels=[0,'',0.4,'',0.8,'',1.2]

#trunc_ticklabels=[0,'',0.5,'',1.0,'']
#full_ticklabels=trunc_ticklabels

perc_steps=[16,50,84]
#bins=np.arange(0,15,0.1)
bins=np.arange(0,20,0.1)


#bins=np.arange(0,15,0.01)
spt.initiate_science_plot()
#plt.figure(figsize=(12,8))
#plt.figure(figsize=(7.25,7.25), constrained_layout=True)
#plt.figure(figsize=(6.,6.), constrained_layout=True)
#plt.figure(figsize=(4.,4.), constrained_layout=True)
#fig_height=1.7/3.*num_plots #This was the method before 2022-09-27 when I tried to put the plot in ApJ and it said it was too big for a page
#fig_height=1.3/3.*num_plots
fig_height=1./3.*num_plots
#text_pos=[1.25,1.05] # #This was the method before 2022-09-27 when I tried to put the plot in ApJ and it said it was too big for a page
text_pos=[1.25,1.]

print('fig_height',fig_height)


#spt.start_ApJ_fig(width_cols=1,constrained_layout=True, width_height=[1.,1.7])
spt.start_ApJ_fig(width_cols=1,constrained_layout=True, width_height=[1.,fig_height])

#count=num_plots
#count=1
#for row in wd_abund_table:
for count in reversed(range(1,num_plots+1)):
    row=wd_abund_table[count-1]
    print('-----\n\n-----\n',row['name'])
    print(row['display_name'])
    if count==1:
        ax1=plt.subplot(num_plots,1,count)
        ax1.set_xticklabels([])
        ax1.set_yticklabels(full_ticklabels)
        #ax1.xaxis.set_tick_params(labelbottom=True)
        plt.xlim(xlims)
        plt.ylim(0,1.2)
    elif count==num_plots:
        #axn=plt.subplot(num_plots,1,count,sharex=ax1)
        axlast=plt.subplot(num_plots,1,count)
        plt.xlim(xlims)
        axlast.set_xticklabels(full_x_ticklabels)
        axlast.set_yticklabels(trunc_ticklabels)
    else:
        #axn=plt.subplot(num_plots,1,count,sharex=ax1)
        axn=plt.subplot(num_plots,1,count)
        #plt.axhline(y=1,color='r',linestyle='-')
        #plt.axhline(y=0.5,color='b',linestyle='--')
        plt.xlim(xlims)
        axn.set_xticklabels([])
        axn.set_yticklabels(trunc_ticklabels)
    print(os.getcwd())
    age_dist=np.genfromtxt(row['age_dist_file'])
    print(row['age_dist_file'],np.nanmax(age_dist))
    print(row['plot_color'],row['plot_color'].replace('"',''))
    plt.hist(age_dist, bins=bins,density=True, color=row['plot_color'].replace('"',''))
    low_bar=np.nanpercentile(age_dist, 16)
    high_bar=np.nanpercentile(age_dist,84)
    med_point=np.nanmedian(age_dist)
    print(low_bar,med_point,high_bar)
    lo_err=med_point-low_bar
    hi_err=high_bar-med_point
    print(med_point,'-',lo_err,'+',hi_err)
    plt.errorbar(med_point, ypos,xerr=[[lo_err],[hi_err]],color=row['plot_color'].replace('"',''),marker=wd_marker,markersize=wd_size,linestyle='None')
    #plot_ml(wd_abund_table.loc['WDJ2317+1830'], ypos=ypos)
    #plt.xlabel('Total Age (Gyr)')
    #plt.text(text_pos[0],text_pos[1],fs.fix_display_string(row['name']))
    plt.text(text_pos[0],text_pos[1],fs.fix_display_string(row['display_name']))
    #plt.text(low_bar,0.8,str(low_bar),rotation=90.)
    #plt.text(high_bar,0.8,str(high_bar),rotation=90.)
    #plt.text(med_point,0.8,str(med_point),rotation=90.)



    plt.axvline(13.8, color='k',linestyle=':')
    #plt.axvline(10.8,color='r',linestyle='--')
    #plt.axvline(low_bar, color='r',linestyle=':')
    #plt.axvline(high_bar, color='r',linestyle=':')
    #plt.axvline(med_point, color='r',linestyle=':')
    #if count==1:
        #ax1.set_yticklabels(full_ticklabels)
    #else:
        #axn.set_yticklabels(trunc_ticklabels)
    if count==num_plots:
        #axn.set_xticklabels([full_tick_labels])
        plt.xlabel('Total Age (Gyr)')
    else:
        #axn.set_xticklabels([])
        pass
    plt.ylim(0,1.2)
    
    if count==num_plots//2:
        plt.ylabel('Probability Density')
        print('ylabel going on',count)
    else:
        pass
    #count+=1
    #count-=1

################################


#plt.axvline(13.8, color='k',linestyle=':')
#####ax1.set_xticklabels([])
#ax1.get_shared_x_axes().join(ax1,ax3)
#ax2.get_shared_x_axes().join(ax2,ax3)
#ax1.set_xticklabels([])
#ax2.set_xticklabels([])
######ax2.set_yticklabels([0,0.2,0.4,0.6,0.8,1.0])
#ax2.set_yticklabels(trunc_ticklabels)
#ax1.get_shared_y_axes().join(ax1,ax3)
######ax1.set_yticklabels([0,0.2,0.4,0.6,0.8,1.0,1.2])
#ax1.set_yticklabels(full_ticklabels)
#plt.subplots_adjust(hspace=0)



#plt.ylabel('Probability Density')
ax1.set_xticklabels([])

plt.subplots_adjust(hspace=0, left=0.17,right=0.97, top=0.95,bottom=0.1)
#plt.xlim(0,14)
#plt.xlim(6,14)
#plt.xlim(6,10)
#plt.text(0,0,'J1644')
#plt.ylim(0,1.2)


print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]

plt.savefig('figure_totalagehist_'+time_string+'.pdf')#plt.grid(True)



plt.show()

fig_height 2.0
-----

-----
 SDSSJ1636+1619
SDSS J1636+1619
/Users/BenKaiser/Desktop/radial_velocity_calculations


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:86: UserWarning: FixedFormatter should only be used together with FixedLocator
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:87: UserWarning: FixedFormatter should only be used together with FixedLocator


1663872696_SDSSJ1636p1619_bedard2020_tot_age_MC_6thick_2p5thin.csv 19.999984066054743
"#ff33f6" #ff33f6
9.257144844494048 10.68520607915142 12.386943704356307
10.68520607915142 - 1.4280612346573722 + 1.7017376252048866
-----

-----
 WDJ2356-209
WD J2356$-$209
/Users/BenKaiser/Desktop/radial_velocity_calculations


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:95: UserWarning: FixedFormatter should only be used together with FixedLocator


1663870024_WDJ2356m209_bedard2020_tot_age_MC_6thick_2p5thin.csv 19.99996246235282
"#00ffc5" #00ffc5
10.633945131914325 12.277041603346735 15.846080922286973
12.277041603346735 - 1.6430964714324094 + 3.569039318940238
-----

-----
 LHS2534
LHS 2534
/Users/BenKaiser/Desktop/radial_velocity_calculations
1663868333_LHS2534_bedard2020_tot_age_MC_6thick_2p5thin.csv 19.99995527685575
"#0059b1" #0059b1
7.572519634413485 8.60569674442757 10.371943958025598
8.60569674442757 - 1.0331771100140852 + 1.766247213598028
-----

-----
 WDJ2317+1830
WD J2317+1830
/Users/BenKaiser/Desktop/radial_velocity_calculations
1663863877_WDJ2317p1830_bedard2020_tot_age_MC_6thick_2p5thin.csv 15.894064038460435
"#ff8d00" #ff8d00
7.728195823031445 9.704138935892278 10.251350019072577
9.704138935892278 - 1.9759431128608327 + 0.547211083180299
ylabel going on 3
-----

-----
 SDSSJ1330+6435
SDSS J1330+6435
/Users/BenKaiser/Desktop/radial_velocity_calculations
1663862037_SDSSJ1330p6435_bedard2020_tot_age_MC_6thick_2p5thin

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:78: UserWarning: FixedFormatter should only be used together with FixedLocator


1663860756_WDJ1644m0449_bedard2020_tot_age_MC_6thick_2p5thin.csv 19.99999829267339
"#ff0000" #ff0000
10.46311626088652 12.158052601691896 14.516232555942885
12.158052601691896 - 1.6949363408053753 + 2.358179954250989
/Users/BenKaiser/Desktop/radial_velocity_calculations
/Users/BenKaiser/Desktop
1664296917.186649


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:165: UserWarning: This figure was using constrained_layout==True, but that is incompatible with subplots_adjust and or tight_layout: setting constrained_layout==False. 
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:184: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


# Above is outputs

###Cut code

###This is where the cut code would have belonged
ax3=plt.subplot(3,1,3)
plt.hist(J2356_mc, bins=bins,density=True, color="#1B01F5")
low_bar=np.nanpercentile(J2356_mc, 16)
high_bar=np.nanpercentile(J2356_mc,84)
med_point=np.nanmedian(J2356_mc)
print(low_bar,med_point,high_bar)
#plot_ml(wd_abund_table.loc['WDJ2317+1830'], ypos=ypos)
plt.xlabel('Total Age (Gyr)')
plt.text(text_pos[0],text_pos[1],fs.fix_display_string('LHS 2534'))
plt.text(low_bar,0.8,str(low_bar),rotation=90.)
plt.text(high_bar,0.8,str(high_bar),rotation=90.)
plt.text(med_point,0.8,str(med_point),rotation=90.)



plt.axvline(13.8, color='k',linestyle=':')
plt.axvline(low_bar, color='r',linestyle=':')
plt.axvline(high_bar, color='r',linestyle=':')
plt.axvline(med_point, color='r',linestyle=':')



ax2=plt.subplot(3,1,2, sharey=ax3)
#plt.hist(J1330_mc, bins=bins, normed=True, color=color_dict['SDSSJ1330+6435'])

#plot_ml(wd_abund_table.loc['SDSSJ1330+6435'], ypos=ypos)


#plt.text(text_pos[0],text_pos[1],fs.fix_display_string('SDSS J1330+6435'))

plt.ylabel('Probability Density')
plt.axvline(13.8, color='k',linestyle=':')
ax1=plt.subplot(3,1,1)

#plt.hist(J1644_mc, bins=bins, normed=True, color=color_dict['GaiaJ1644-0449'])

#plot_ml(wd_abund_table.loc['GaiaJ1644-0449'],ypos=ypos)
#plt.text(text_pos[0],text_pos[1],fs.fix_display_string('Gaia J1644-0449'))




In [11]:
print(wd_abund_table[1])

     name        display_name  modeler cooling_model teff teff_err logg logg_err m_wd  h/he h/he_err li/he li/he_err be/he be/he_err na/he na/he_err mg/he mg/he_err k/he k/he_err ca/he ca/he_err cr/he cr/he_err fe/he fe/he_err li/ca li/ca_err na/ca na/ca_err mg/ca mg/ca_err k/ca k/ca_err cr/ca cr/ca_err fe/ca fe/ca_err li/na  li/na_err  k/na k/na_err ca/na ca/na_err li/k li/k_err na/k na/k_err ca/fe ca/fe_err mg/fe mg/fe_err na/mg na/mg_err ca/mg ca/mg_err ca/cr ca/cr_err k/cr k/cr_err cr/fe cr/fe_err na/li  na/li_err  ca/li ca/li_err atm_type diff_atm_type log_q     age      age_minus    age_plus    med_age   med_age_minus med_age_plus  vtan_lsr vtan_lsr_err_lo vtan_lsr_err_hi      v           uw2       v_err_lo  v_err_hi uw2_err_lo  uw2_err_hi plot_color                           age_dist_file                            show show_li_evo show_geo show_dp thin_disk thick_disk halo show_age_dist
-------------- --------------- ------- ------------- ---- -------- ---- -------- ---- ----- 